# TIPS PEALD: frozen STEP 1 + STEP 2, train target readout only

All six source encoders and the complete source backbone, including text projection and Cross-attention, stay frozen. No encoder/Fusion retraining or per-target-Step source search. The default source encoder is DEP.; the other five remain loaded and frozen. Final features are cached once, then only the target answer head trains.

The local v9 checkpoint uses an explicit legacy channel-independent input adapter for unseen sensors. Its weights are unchanged; this is not native joint-sensor transfer. Test scores describe paired synthetic changes, not equipment-fault accuracy. See `_docs/tips-peald-training.md`.


In [ ]:
from pathlib import Path
import sys, os, json
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"
cwd=Path.cwd().resolve()
ROOT=next(p for p in [cwd,cwd/"GEVIS",*cwd.parents] if (p/"src/pecvd/experiments/tips_training.py").is_file())
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
import importlib
from pecvd.experiments import tips_dataset, tips_profiles, tips_optimization, tips_model, tips_training, tips_frozen_source, tips_frozen_training
for module in [tips_dataset,tips_profiles,tips_optimization,tips_model,tips_training,tips_frozen_source,tips_frozen_training]: importlib.reload(module)
run=tips_frozen_training.run
CONFIG=json.loads((ROOT/"config/experiments/tips_peald.json").read_text(encoding="utf-8"))
DEVICE="cuda"  # Explicitly change to cpu for a CPU run.
OUTPUT=None  # Or choose a new experiment directory.
RUN_TRAINING=True  # Set False to evaluate an existing matching checkpoint.
SELECTION=tips_profiles.resolve_config(CONFIG,ROOT)
print({"selected_steps":SELECTION["main_steps"],"selected_sensors":SELECTION["selected_sensors"],"input_count":len(SELECTION["selected_sensors"])})


## 1. Verify frozen source bundle, profiles and Train windows
All six encoder hashes must match the STEP 1 manifest and the STEP 2 binding. This step reads Train measurements only. Source architecture and patch width come from the checkpoints.


In [ ]:
run(CONFIG,ROOT,action="prepare",device="cpu",output=OUTPUT)


## 2. Extract fixed Train/Validation features once, then train the readout
Every STEP 1/2 tensor remains frozen and in eval mode. Only the answer head is supplied to the optimizer. Features are stored in memory-mapped files and reused across epochs and matching reruns. The initial feature extraction still takes time; batch progress and `progress.json` distinguish it from readout fitting. The head fitting time budget excludes feature extraction.
After updating, interrupt the old cell, restart the kernel, and run from the top. Scratch-run weights are not resumed.


In [ ]:
if RUN_TRAINING:
    run(CONFIG,ROOT,action="train",device=DEVICE,output=OUTPUT)


## 3. Extract held-out Test features and evaluate the trained readout
Test is never used for scaling, training or model selection. Report BOOLEAN, CHOICE and ORDINAL separately. Cached-head timing excludes the fixed source forward and is not full inference latency.


In [ ]:
REPORT=run(CONFIG,ROOT,action="evaluate",device=DEVICE,output=OUTPUT)
display(REPORT["metrics"])
